In [17]:
import os
import warnings
warnings.filterwarnings("ignore")
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scanpy.external as sce

In [18]:
in_dir = "../../results/02_10x_visium/04_remove_isolated_spots"
out_dir = "../../results/02_10x_visium/05_integrate_data"
os.makedirs(out_dir, exist_ok=True)

In [19]:
sample_list = ['15837', '15852', '16512', '21022', '8262', 
               'B23-03794-5', 'B23-05785-7-1', 'B23-05785-7-2', 
               'X1842', 'X1845', 'X1899', 'X1912']

In [20]:
df_count_list, df_obs = [], []
for sample in sample_list:
    print(f"Processing sample {sample}...")
    adata = sc.read_h5ad(f"{in_dir}/{sample}.h5ad")
    df_count_list.append(adata.to_df())
    df_obs.append(adata.obs)

df_counts = pd.concat(df_count_list)
df_obs = pd.concat(df_obs)
df_var = adata.var.copy()

Processing sample 15837...
Processing sample 15852...
Processing sample 16512...
Processing sample 21022...
Processing sample 8262...
Processing sample B23-03794-5...
Processing sample B23-05785-7-1...
Processing sample B23-05785-7-2...
Processing sample X1842...
Processing sample X1845...
Processing sample X1899...
Processing sample X1912...


In [21]:
adata = ad.AnnData(X=df_counts.values, obs=df_obs, var=df_var)
adata.var_names_make_unique()
adata.obs_names_make_unique()

In [22]:
# add group info to adata.obs based on sample column in adata.obs
group_dict = {
    'sample_15837': 'SRC',
    'sample_15852': 'SRC',
    'sample_16512': 'SRC',
    'sample_21022': 'SRC',
    'sample_8262': 'SRC',
    'sample_B23-03794-5': 'Normal',
    'sample_B23-05785-7-1': 'Normal',
    'sample_B23-05785-7-2': 'Normal',
    'sample_X1842': 'LN',
    'sample_X1845': 'LN',
    'sample_X1899': 'LN',
    'sample_X1912': 'LN'
}
adata.obs['group'] = adata.obs['sample'].map(group_dict)

In [23]:
adata.obs.head()

,in_tissue,array_row,array_col,sample,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,group
AACCAAGGTATCAGGC-1,1.0,38.0,104.0,sample_15837,7183,8.879612,23835.0,10.078953,16.731697,21.875393,28.336480,40.054542,SRC
AACCGCCAGACTACTT-1,1.0,39.0,45.0,sample_15837,2865,7.960673,5314.0,8.578288,24.049680,29.318781,36.206248,49.642454,SRC
AACGAATTGACCGGTT-1,1.0,35.0,39.0,sample_15837,3985,8.290544,7695.0,8.948456,16.530214,21.247563,28.161144,41.845354,SRC
AACTCAACCTTGACCA-1,1.0,39.0,79.0,sample_15837,4318,8.370779,8361.0,9.031453,19.973687,23.992345,29.840928,41.813180,SRC
AACTGCCTCGATAGGT-1,1.0,37.0,61.0,sample_15837,5434,8.600615,10826.0,9.289799,12.848698,16.709773,22.464437,34.435618,SRC


In [24]:
adata.write_h5ad(f'{out_dir}/adata.h5ad')

In [25]:
df_counts = adata.to_df().astype(int)

In [26]:
df_counts.to_csv(f'{out_dir}/counts_matrix.csv')
adata.obs.to_csv(f'{out_dir}/obs.csv')
adata.var.to_csv(f'{out_dir}/var.csv')